<a href="https://colab.research.google.com/github/Kotyga/ML_toolkit/blob/main/%D0%AD%D0%BC%D0%B1%D0%B5%D0%B4%D0%B4%D0%B8%D0%BD%D0%B3%D0%B8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ДЗ 2. Эмбеддинги токенов и позиций

Цель работы — реализовать два модуля PyTorch, которые формируют входные представления трансформера: `TokenEmbeddings` и `PositionalEmbeddings`.

Заполните каркасы классов и выполните проверки после каждого задания. В этой работе используются обучаемые абсолютные позиционные эмбеддинги. Реализовывать механизм внимания или обучать языковую модель не требуется.

Обозначения: `batch_size` — число примеров, `seq_len` — длина последовательности, `emb_size` — размерность вектора.

## Подготовка

Необходим Python 3 и PyTorch. Если PyTorch ещё не установлен в окружении ядра, раскомментируйте следующую строку установки. Для выполнения достаточно CPU.

In [ ]:
# %pip install torch

In [ ]:
import torch
from torch import nn

torch.manual_seed(42)

## Задание 1. Token Embeddings

Создайте класс `TokenEmbeddings`, наследующийся от `nn.Module`.

**Конструктор** принимает:

- `vocab_size: int` — размер словаря токенизатора;
- `emb_size: int` — размерность эмбеддингов.

В конструкторе создайте таблицу `nn.Embedding` размера `vocab_size × emb_size` и сохраните её в атрибуте `self.embedding`.

**Метод `forward(x)`** принимает тензор целочисленных ID формы `batch_size × seq_len`. Он извлекает соответствующие строки таблицы в порядке входных ID и возвращает тензор формы `batch_size × seq_len × emb_size`.

Пример входа:

```text
tensor([[113, 456,  76, 345],
        [345, 678, 454, 546]])
```

Одинаковому ID должен соответствовать один и тот же вектор независимо от позиции и номера примера в батче.

In [ ]:
class TokenEmbeddings(nn.Module):
    def __init__(self, vocab_size: int, emb_size: int):
        super().__init__()
        # TODO: создайте self.embedding.
        raise NotImplementedError("Реализуйте создание таблицы эмбеддингов токенов")

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # TODO: верните векторы для входных ID с сохранением формы батча и порядка.
        raise NotImplementedError("Реализуйте выбор векторов токенов")

### Проверка задания 1

Проверяются форма результата, соответствие ID строкам таблицы и одинаковые представления повторяющихся токенов. Ошибка `NotImplementedError` до заполнения каркаса ожидаема.

In [ ]:
vocab_size, emb_size = 1000, 16
x = torch.tensor([[113, 456, 76, 345],
                  [345, 678, 454, 546]], dtype=torch.long)

token_embeddings = TokenEmbeddings(vocab_size, emb_size)
token_vectors = token_embeddings(x)

assert token_vectors.shape == (2, 4, emb_size)
assert token_embeddings.embedding.weight.shape == (vocab_size, emb_size)
assert token_vectors.is_floating_point()
for b in range(x.shape[0]):
    for t in range(x.shape[1]):
        torch.testing.assert_close(token_vectors[b, t], token_embeddings.embedding.weight[x[b, t]])
torch.testing.assert_close(token_vectors[0, 3], token_vectors[1, 0])
print("TokenEmbeddings: проверки пройдены.")

## Задание 2. Positional Embeddings

Создайте класс `PositionalEmbeddings`, наследующийся от `nn.Module`.

**Конструктор** принимает:

- `max_seq_len: int` — число доступных позиций, то есть максимальную длину последовательности;
- `emb_size: int` — размерность эмбеддингов.

Создайте таблицу `nn.Embedding` размера `max_seq_len × emb_size` и сохраните её в `self.embedding`.

**Метод `forward(seq_len)`** принимает длину последовательности `seq_len: int`. Извлеките векторы позиций от `0` до `seq_len` **не включительно** и верните тензор формы `seq_len × emb_size`.

В проверках используются длины в диапазоне `1 ≤ seq_len ≤ max_seq_len`. Создавайте индексы позиций на том же устройстве, где находится таблица. ID токенов этому модулю не передаются.

In [ ]:
class PositionalEmbeddings(nn.Module):
    def __init__(self, max_seq_len: int, emb_size: int):
        super().__init__()
        # TODO: создайте self.embedding.
        raise NotImplementedError("Реализуйте создание таблицы позиционных эмбеддингов")

    def forward(self, seq_len: int) -> torch.Tensor:
        # TODO: создайте индексы позиций и верните соответствующие векторы.
        raise NotImplementedError("Реализуйте выбор векторов позиций")

### Проверка задания 2

Проверяются форма, порядок позиционных векторов и согласованность результатов для разных длин последовательности.

In [ ]:
max_seq_len = 32
positional_embeddings = PositionalEmbeddings(max_seq_len, emb_size)
position_vectors = positional_embeddings(x.shape[1])

assert positional_embeddings.embedding.weight.shape == (max_seq_len, emb_size)
assert position_vectors.shape == (4, emb_size)
for t in range(4):
    torch.testing.assert_close(position_vectors[t], positional_embeddings.embedding.weight[t])
torch.testing.assert_close(positional_embeddings(2), position_vectors[:2])
assert positional_embeddings(1).shape == (1, emb_size)
assert positional_embeddings(max_seq_len).shape == (max_seq_len, emb_size)
print("PositionalEmbeddings: проверки пройдены.")

## Проверка совместимости

Векторы токенов имеют форму `batch_size × seq_len × emb_size`, а векторы позиций — `seq_len × emb_size`. Выполните их поэлементное сложение и сохраните результат в `input_embeddings`. Размерность эмбеддингов у обоих модулей одинакова; позиционные векторы общие для всех примеров батча.

In [ ]:
# TODO: сложите token_vectors и position_vectors.
input_embeddings = ...

In [ ]:
assert isinstance(input_embeddings, torch.Tensor)
assert input_embeddings.shape == token_vectors.shape
for b in range(x.shape[0]):
    torch.testing.assert_close(
        input_embeddings[b] - token_vectors[b], position_vectors, atol=1e-6, rtol=1e-5
    )
print("Форма входа трансформера:", tuple(input_embeddings.shape))

## Вопросы по результатам работы

1. Почему ID используется как индекс таблицы, а не как числовая характеристика токена?
2. Почему повторяющийся токен получает одинаковый входной эмбеддинг?
3. Почему результат `PositionalEmbeddings.forward` не содержит измерения батча?
4. Как при сложении сохраняется форма `batch_size × seq_len × emb_size`?
5. Какие параметры этих модулей будут обновляться при последующем обучении модели?

Сохраните ноутбук с реализациями, результатами проверок и краткими ответами.